<a href="https://colab.research.google.com/github/narpavi-ai/lasp-agent-workshop/blob/main/notebooks/02-mcp-and-agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 · MCP, and an agent

**LASP workshop: build your agent**

Same question as Lab 1. This time the model can use the LASP Isotherm
Fitter and IAST apps as **tools**, served by an **MCP server**. The tools
run the apps' own engines, unchanged.

1. **What MCP is**, and connecting to the LASP server.
2. **You are the workflow:** you call the tools yourself, step by step.
3. **An agent:** the model decides the steps.

About **12 requests**.

## Before you start

Add these with the **🔑 key icon** in Colab's left sidebar, and turn **Notebook access** on for each. Notebook access is per notebook, so each lab asks again.

| Name | Value |
|---|---|
| `GOOGLE_API_KEY` | your free key from <https://aistudio.google.com/apikey> |
| `LASP_MCP_TOKEN` | the workshop token, shown on the slide |
| `ANTHROPIC_API_KEY` | *optional:* only if you switch to Claude (a paid key from <https://console.anthropic.com>) |

## Setup

Run these in order. Each takes a few seconds.

### Setup · Install

LangChain talks to the model, LangGraph runs agents, and the MCP adapter connects to the LASP server. Exact versions, as tested, plus this workshop's helper file.

In [ ]:
# pip may print a warning about google-auth and google-colab. That's expected
# and harmless: Gemini's library needs a newer google-auth than Colab ships.
%pip install -q langchain==1.4.7 langchain-core==1.7.0 langchain-google-genai==4.4.2 langchain-anthropic==1.7.6 langgraph==1.2.17 langchain-mcp-adapters==0.3.2 mcp==1.30.0
!curl -sSfO https://raw.githubusercontent.com/narpavi-ai/lasp-agent-workshop/main/notebooks/lasp_lab.py
print("✅ Installed, and helpers downloaded.")

### Setup · Helpers

Display and connection helpers, so the notebook shows only what's being taught. Curious? Read [`lasp_lab.py`](lasp_lab.py).

In [ ]:
from lasp_lab import *   # show(), call(), run_live() and friends: see lasp_lab.py
print("✅ Helpers ready.")

### Setup · Choose your model

`"gemini"` is free and needs `GOOGLE_API_KEY`. `"claude"` is paid and needs `ANTHROPIC_API_KEY`. Nothing else in the lab changes.

In [ ]:
PROVIDER = "gemini"    # "gemini" or "claude"

### Setup · Load your key

Read from Colab Secrets, so it never appears in the notebook.

In [ ]:
import os
KEY_NAME = {"gemini": "GOOGLE_API_KEY", "claude": "ANTHROPIC_API_KEY"}[PROVIDER]
os.environ[KEY_NAME] = secret(KEY_NAME)
print(f"✅ {KEY_NAME} loaded.")

### Setup · Create the model

One line picks the provider and model. Everything after it is the same for both.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.rate_limiters import InMemoryRateLimiter

MODELS = {"gemini": "google_genai:gemini-3.5-flash-lite", "claude": "anthropic:claude-haiku-5-5"}
MODEL = MODELS[PROVIDER]

# Gemini's free tier allows 15 requests a minute; stay under it. Paid Claude keys allow more.
limiter = InMemoryRateLimiter(requests_per_second=0.2) if PROVIDER == "gemini" else None

# Claude 5 thinks before answering. Low effort keeps that short (faster, cheaper),
# and the cap leaves room for the answer after the thinking.
extra = {"max_tokens": 16000, "thinking": {"type": "adaptive"}, "output_config": {"effort": "low"}} if PROVIDER == "claude" else {}
model = init_chat_model(MODEL, rate_limiter=limiter, **extra)
print(f"✅ Model ready: {MODEL}")

### Step 1 · What is MCP?

The **Model Context Protocol** is a standard way to give AI models tools.
A **server** offers tools; any **client** can use them: this notebook,
n8n, Claude Desktop, an IDE. Write the tools once, use them everywhere.
It's often described as "USB-C for AI tools".

The LASP server runs on Google Cloud. To connect you need two things:

| Placeholder | What it is | Where it comes from |
|---|---|---|
| `LASP_MCP_URL` | the server's address | already filled in below |
| `LASP_MCP_TOKEN` | the workshop password | your Colab secret (the token on the slide) |

An MCP server **describes its own tools**. Each description is what the
model reads when it decides which tool to use.

In [ ]:
LASP_MCP_URL = "https://lasp-mcp-189650845754.northamerica-northeast1.run.app/mcp"   # where the LASP server lives
LASP_MCP_TOKEN = secret("LASP_MCP_TOKEN")   # the workshop token, from Colab Secrets

client, tools = await connect(LASP_MCP_URL, LASP_MCP_TOKEN)

for t in tools:
    print(f"{t.name:22s} {t.description[:90]}…")

Eight tools: the Isotherm Fitter (`fit_isotherm`, `plot_isotherm_fit`),
IAST (`iast_binary`, `iast_sweep_summary`, `iast_xy_diagram`, `plot_iast`),
and two helpers. Results also carry `workshop_validation`, physical checks
that are a **workshop addition**, not part of the LASP apps.

### Step 2 · You are the workflow

A tool is just a function on a server; no model needed. Do what you'd do
in the lab: fit both isotherms with all 16 of the Isotherm Fitter's models
(ranked by AICc, as the app ranks them).

In [ ]:
# Synthetic 13X-like data made for this workshop (not measured), 298 K.
# The same numbers are on the server as workshop_co2_298K_synthetic and workshop_n2_298K_synthetic.
pressure_bar = [0.001, 0.003, 0.01, 0.03, 0.1, 0.2, 0.4, 0.6, 0.8, 1, 1.5, 2, 3, 5, 7, 10]
co2_mol_per_kg = [0.147171, 0.399272, 1.02996, 1.9223, 2.88179, 3.25169, 3.86714, 4.16431, 4.24648, 4.41152, 4.66168, 4.89841, 5.10685, 5.2541, 5.17341, 5.23311]
n2_mol_per_kg = [0.000319116, 0.000968288, 0.00320824, 0.00947269, 0.0311209, 0.0631523, 0.122319, 0.186175, 0.241094, 0.29566, 0.436553, 0.552997, 0.78373, 1.14108, 1.43167, 1.77024]

co2_fit = await call("fit_isotherm", pressure=pressure_bar, loading=co2_mol_per_kg)
n2_fit = await call("fit_isotherm", pressure=pressure_bar, loading=n2_mol_per_kg)
show_fit(co2_fit, "CO₂ fit")
show_fit(n2_fit, "N₂ fit")
await call("plot_isotherm_fit", pressure=pressure_bar, loading=n2_mol_per_kg, panel="fits");

That plot is the app's "Data & fits" panel for N₂: the top five models sit
almost on top of each other. Now take each gas's top-ranked fit and run IAST
for flue gas.

In [ ]:
def top_iast_fit(fit):
    """The rule we chose: the highest-ranked fit that IAST can use."""
    return next(f for f in fit["ranked"] if f.get("iast_handoff"))

co2_model, n2_model = top_iast_fit(co2_fit), top_iast_fit(n2_fit)
print("Chosen:", co2_model["model"], "for CO2,", n2_model["model"], "for N2")

iast = await call("iast_binary", component1=co2_model["iast_handoff"][0], component2=n2_model["iast_handoff"][0],
                  total_pressure_bar=[1], y1=[0.15])
show_iast(iast)
await call("plot_iast", component1=co2_model["iast_handoff"][0], component2=n2_model["iast_handoff"][0],
           panel="loadings_vs_P", y1=0.15);

**A real number, from the real solver,** and no model involved. The yellow
band marks where IAST needs N₂'s isotherm beyond the 10 bar it was fitted
to, which is the extrapolation warning.

This is a **workflow**: fixed steps that you decided in advance (the n8n
track builds the same thing as a chain of nodes). It's predictable, and it
does exactly what you wrote. Notice what the rule did: it picked **Toth**
for N₂ (S ≈ 414). But Toth, Quadratic and Langmuir are within 1.5 AICc
units, a statistical tie, and give S = 414, 420 and 388. The workflow
can't notice that, because you didn't write a step for it.

### Step 3 · An agent

An agent replaces your fixed steps with a **loop**: the model either calls
a tool or answers. `create_agent` builds it from a model, the tools, and a
**system prompt**, the agent's standing instructions.

In [ ]:
from langchain.agents import create_agent

agent_tools = [t for t in tools if not t.name.startswith("plot_")]   # images cost tokens
SYSTEM_PROMPT = ("You are a research assistant at the Laboratory for Advanced Separation Processes (LASP), "
         "University of Alberta, with the LASP Isotherm Fitter and IAST tools. "
         "Never estimate a number a tool can compute. Report the models you used and any "
         "workshop_validation warnings.")

agent = create_agent(model, agent_tools, system_prompt=SYSTEM_PROMPT)
draw(agent)

**That's the loop:** `model → tools → model → …` until the model answers
instead of calling a tool. LangGraph runs it.

Now the **user prompt**: Lab 1's question, with the data in it.

In [ ]:
USER_PROMPT = (f"Here are pure-component CO2 and N2 isotherms at 298 K on a 13X-like adsorbent.\n"
               f"pressure (bar): {pressure_bar}\nCO2 loading (mol/kg): {co2_mol_per_kg}\n"
               f"N2 loading (mol/kg): {n2_mol_per_kg}\n\n"
               "Using IAST, what is the CO2/N2 selectivity for a flue gas with 15% CO2 at 1 bar total pressure?")

messages = await run_live(agent, USER_PROMPT)

Read the trace. The model chose every step: which tool, which arguments,
and when to stop. Did it copy your data correctly into `fit_isotherm`?
Which N₂ model did it pick, and did it say why? If a workshop check
flagged a mistake, did it notice and retry?

### Step 4 · A question your workflow can't answer

In [ ]:
FOLLOW_UP = (USER_PROMPT + "\n\nSeveral N2 models may fit about equally well. "
             "Does the choice of N2 model change the selectivity? Compare the top three IAST-compatible N2 fits.")
messages = await run_live(agent, FOLLOW_UP)

Your Step 2 code would need new steps for this. The agent planned it from
the question. That flexibility is the point of an agent, and also its
risk: **you didn't decide the steps, so you have to check them.** That's
Labs 3 and 4.

### Step 5 · The safety belt

An agent loops until it decides to stop. `recursion_limit` caps the
steps, so a confused agent can't loop and spend forever.

In [ ]:
from langgraph.errors import GraphRecursionError
try:
    await run_live(agent, FOLLOW_UP, recursion_limit=4)
except GraphRecursionError:
    print("🛑 Stopped by recursion_limit, as intended.")

## What you built

| | Workflow (Step 2) | Agent (Steps 3–4) |
|---|---|---|
| Who decides the steps | you, in code | the model, at run time |
| Model requests | none | one per step |
| Predictable | yes | no, so you evaluate it |
| Handles a new question | only if you wrote steps for it | yes |

**In the n8n track** the workflow is MCP Client nodes in a row, and the
agent is the AI Agent node with an MCP Client Tool.

## If something breaks

| What you see | Fix |
|---|---|
| `404 NOT_FOUND`, model not available | The provider retired the model. Pick a current one (<https://aistudio.google.com> or <https://docs.anthropic.com/en/docs/about-claude/models>) and change it in `MODELS`. |
| `401`, `API key not valid`, `authentication_error` | Check the key's secret (`GOOGLE_API_KEY` or `ANTHROPIC_API_KEY`): exact name, Notebook access on, and that `PROVIDER` matches it. |
| `Couldn't connect to …` | The message says why. A `401` means the `LASP_MCP_TOKEN` secret is wrong: 48 characters, `0-9` and `a-f` (a trailing `%` copied from the terminal isn't part of it). |
| `429`, `RESOURCE_EXHAUSTED` | Free-tier limit. Wait a minute and re-run the cell. |
| `ModuleNotFoundError`, or a name not defined | The runtime restarted. Run the setup cells again from the top. |